# VisClick — Colab setup

Run cells **top to bottom**. Authorise Google Drive when prompted.

- **GitHub** → code at `/content/visclick`
- **Drive** `MyDrive/visclick` → datasets + weights (persists after disconnect)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
if not os.path.isdir("/content/visclick"):
    subprocess.run(["git", "clone", REPO, "/content/visclick"], check=True)
else:
    subprocess.run("cd /content/visclick && git pull --rebase", shell=True, check=False)
%cd /content/visclick


In [ ]:
import subprocess
try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
        print(subprocess.check_output(["nvidia-smi"]).decode())
except Exception as e:
    print("GPU check:", e)


In [ ]:
import os
DRIVE = "/content/drive/MyDrive/visclick"
os.makedirs(DRIVE, exist_ok=True)
for sub in (
    "data/raw", "data/clay", "data/vins", "data/webui", "data/ui_vision",
    "data/unified", "data/source_train", "data/desktop_labeled", "data/desktop_test",
    "weights/baseline_source", "weights/experiments", "videos",
):
    os.makedirs(os.path.join(DRIVE, sub), exist_ok=True)
print("DRIVE =", DRIVE)


In [ ]:
!pip install -q ultralytics==8.* opencv-python "huggingface_hub[cli]>=0.20" tqdm matplotlib pandas


## 03 — Fine-tune on desktop (YOLO)

After CVAT export to `DRIVE/.../data/desktop_labeled` in YOLO format.


In [ ]:
import os
DRIVE = "/content/drive/MyDrive/visclick"
os.environ["VISCLICK_DRIVE"] = DRIVE
!python scripts/patch_colab_configs.py


In [ ]:
from ultralytics import YOLO
import os, time
DRIVE = "/content/drive/MyDrive/visclick"
# Start from source baseline
w = f"{DRIVE}/weights/baseline_source/best_source_v8s.pt"
if not os.path.isfile(w):
    w = "yolov8s.pt"
m = YOLO(w)
t0 = time.time()
m.train(
    data="configs/yolo_desktop_finetune_colab.yaml",
    epochs=120,
    imgsz=640,
    batch=8,
    optimizer="AdamW",
    lr0=5e-4,
    cos_lr=True,
    freeze=10,
    warmup_epochs=3,
    patience=20,
    project=f"{DRIVE}/weights/experiments",
    name="M3_desktop",
    plots=True,
)
print("s:", int(time.time() - t0))
